# 01.7 — ADME MMP Analysis

Recreates the Matched Molecular Pair (MMP) analysis from Fang et al. (2023), §5.4/5.5, on our own 3,521-compound public ADME set (paper used >25,000 internal compounds). Uses [`mmpdb`](https://github.com/rdkit/mmpdb) via `src/mmp/` (see [`src/mmp/CLAUDE.md`](../src/mmp/CLAUDE.md)) to fragment + index the dataset and extract statistically significant transformation rules per endpoint.

**Method (matches the paper, confirmed against mmpdb source):**
- Fragmentation & indexing use mmpdb's own defaults — the paper's quoted cut-SMARTS/heavy-atom/rotatable-bond parameters are literally mmpdb's built-in defaults, not a custom rule set. Pinned explicitly in `src/mmp/mmp.py` so a future mmpdb version can't silently drift.
- "Representative rules" = per-rule statistics with ≥5 matched pairs, paired-t-test p<0.05, at the most specific environment radius ≤3 — mmpdb always computes radius 0–5 at index time; the paper's "max radius 3" is a query-time selection, reimplemented in `significant_rules()`.

**Scope**: straight recreation only (single run on the full dataset). A data-quantity ablation (subsampling to see how the significant-rule count degrades with N, mirroring the project's learning-curve experiments) and a parameter-sensitivity pass are deferred.

## 1 — Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from src.mmp import write_smi_file, write_properties_file, run_fragment, run_index, significant_rules

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
MMP_DIR = DATA_PROC / 'mmp'
MMP_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINTS = {
    'HLM':  'LOG HLM_CLint (mL/min/kg)',
    'MDR1': 'LOG MDR1-MDCK ER (B-A/A-B)',
    'SOL':  'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'RLM':  'LOG RLM_CLint (mL/min/kg)',
}

## 2 — Load data and write mmpdb input files

`mmpdb`'s property-file format requires short, whitespace-free property names, so we rename the endpoint columns to their short forms before writing.

In [ ]:
df = pd.read_csv(DATA_RAW / 'ADME_public_set_3521.csv')
df = df.rename(columns={long: short for short, long in ENDPOINTS.items()})
print(df.shape)
df[['Internal ID', 'SMILES'] + list(ENDPOINTS.keys())].head()

In [ ]:
smi_path = MMP_DIR / 'adme.smi'
props_path = MMP_DIR / 'adme_props.csv'

write_smi_file(df, smiles_col='SMILES', id_col='Internal ID', path=smi_path)
write_properties_file(df, id_col='Internal ID', property_cols=list(ENDPOINTS.keys()), path=props_path)

## 3 — Fragment + index

Builds the MMP SQLite database. Takes well under a minute on this dataset size.

In [ ]:
fragments_path = MMP_DIR / 'adme.fragments'
db_path = MMP_DIR / 'adme.mmpdb'

t0 = time.time()
run_fragment(smi_path, fragments_path, num_jobs=8)
print(f'fragment: {time.time() - t0:.1f}s')

t0 = time.time()
run_index(fragments_path, props_path, db_path)
print(f'index: {time.time() - t0:.1f}s')

## 4 — Database scale vs the paper

The paper built its database from >25,000 internal compounds and reported >1.2M rules. Checking how our 3,521-compound public set compares before filtering for significance.

In [ ]:
con = sqlite3.connect(db_path)
n_compounds_indexed = con.execute('select count(*) from compound').fetchone()[0]
n_pairs = con.execute('select count(*) from pair').fetchone()[0]
n_candidate_rules = con.execute('select count(*) from rule').fetchone()[0]
con.close()

print(f'compounds indexed (>= 1 non-missing property): {n_compounds_indexed} / {len(df)}')
print(f'matched pairs: {n_pairs}')
print(f'candidate rules (pre-significance-filter): {n_candidate_rules}')
print('paper: >25,000 compounds -> >1.2M rules')

### 4.1 — Why 5181 candidate rules doesn't mean 5181 usable rules per endpoint

`rule` (5181, above) and `pair` (31938) are purely structural facts — "this V1→V2 transformation occurs somewhere in the indexed compounds" — computed with no knowledge of HLM/MDR1/SOL/RLM specifically. A rule only becomes usable for *one* endpoint's statistics once mmpdb can find matched pairs where both molecules have a non-missing value for that endpoint.

Walking that funnel by hand for one endpoint (HLM) first, then generalizing to all four below.

In [ ]:
def count_rules_with_stats(con, property_id, min_pairs=0, max_p_value=None):
    """Count distinct rules with a rule_environment_statistics row (radius<=3, count>=min_pairs) for this property."""
    query = '''select count(distinct r.id)
               from rule r
               join rule_environment re on re.rule_id = r.id
               join rule_environment_statistics res on res.rule_environment_id = re.id
               where res.property_name_id = ? and re.radius <= 3 and res.count >= ?'''
    params = [property_id, min_pairs]
    if max_p_value is not None:
        query += ' and res.p_value < ?'
        params.append(max_p_value)
    return con.execute(query, params).fetchone()[0]

In [ ]:
con = sqlite3.connect(db_path)
hlm_property_id = con.execute('select id from property_name where name = ?', ('HLM',)).fetchone()[0]

n_hlm_raw = df['HLM'].notna().sum()
print(f'raw dataset: {n_hlm_raw} / {len(df)} compounds have an HLM value')

n_hlm_indexed_with_property = con.execute(
    'select count(distinct compound_id) from compound_property where property_name_id = ?',
    (hlm_property_id,),
).fetchone()[0]
print(f'of the {n_compounds_indexed} indexed compounds (those with >=1 structural match), '
      f'{n_hlm_indexed_with_property} still have an HLM value')

n_hlm_any_stat = count_rules_with_stats(con, hlm_property_id)
print(f'of the {n_candidate_rules} candidate rules, only {n_hlm_any_stat} have >=1 matched pair '
      f'where both molecules have HLM data')

n_hlm_min_pairs = count_rules_with_stats(con, hlm_property_id, min_pairs=5)
print(f"of those, only {n_hlm_min_pairs} clear the paper's >=5-matched-pair minimum")

n_hlm_significant = count_rules_with_stats(con, hlm_property_id, min_pairs=5, max_p_value=0.05)
print(f'and {n_hlm_significant} of those also pass p<0.05 -- this is the HLM row shown in §5 below')

con.close()

### 4.2 — Same funnel, all four endpoints

Reusing `count_rules_with_stats` from 4.1 for each endpoint.

In [ ]:
con = sqlite3.connect(db_path)

funnel_rows = []
for ep in ENDPOINTS:
    property_id = con.execute('select id from property_name where name = ?', (ep,)).fetchone()[0]
    n_indexed_with_property = con.execute(
        'select count(distinct compound_id) from compound_property where property_name_id = ?',
        (property_id,),
    ).fetchone()[0]
    funnel_rows.append({
        'endpoint': ep,
        'compounds with property': n_indexed_with_property,
        'rules with >=1 matched pair': count_rules_with_stats(con, property_id),
        'rules with >=5 pairs': count_rules_with_stats(con, property_id, min_pairs=5),
        '+ p<0.05': count_rules_with_stats(con, property_id, min_pairs=5, max_p_value=0.05),
    })

con.close()

funnel_df = pd.DataFrame(funnel_rows).set_index('endpoint')
funnel_df

## 5 — Representative rules per endpoint

Filter: ≥5 matched pairs, paired-t-test p<0.05, most specific environment radius ≤3 — exactly the paper's stated criteria (§5.4).

In [ ]:
rules_by_endpoint = {}
for ep in ENDPOINTS:
    rules = significant_rules(db_path, property_name=ep, max_radius=3, min_pairs=5, max_p_value=0.05)
    rules_by_endpoint[ep] = rules
    print(f'{ep}: {len(rules)} significant rules')

In [ ]:
for ep, rules in rules_by_endpoint.items():
    print(f'--- {ep} ---')
    display(rules[['from_smiles', 'to_smiles', 'n_pairs', 'mean_change', 'std_change', 'p_value']])

## 6 — Summary: significant rules per endpoint

In [ ]:
counts = pd.Series({ep: len(rules) for ep, rules in rules_by_endpoint.items()})

fig, ax = plt.subplots(figsize=(5, 3.5))
counts.plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('significant rules (>=5 pairs, p<0.05, radius<=3)')
ax.set_xlabel('endpoint')
ax.set_title('MMP rules surviving significance filter, per endpoint')
plt.tight_layout()
plt.savefig('../figures/section5_mmp_significant_rules_per_endpoint.png', dpi=150)
plt.show()

## 7 — Discussion

The mmpdb pipeline itself recreates cleanly — fragmentation and indexing use mmpdb's own defaults (confirmed to match the paper's stated parameters verbatim by reading the mmpdb source), and the significance filter (≥5 pairs, p<0.05, radius≤3) is a direct SQL reimplementation of what mmpdb computes internally.

What differs sharply is yield: the paper's >25,000-compound internal set produced >1.2M rules; our 3,521-compound public set produces on the order of ~5,000 *candidate* rules (§4), and only a small handful survive the significance filter per endpoint (§5/§6) — most transformations here simply never accumulate 5 matched pairs. This isn't a bug in the recreation, it's the expected consequence of a ~7x smaller compound set combined with a strict per-transform pair-count requirement.

That sparsity is itself informative for this project: MMP rule discovery is highly data-hungry, more so than the ML models studied elsewhere in this project (which still show signal at much smaller training fractions in the learning-curve experiments). A natural follow-up — deferred for now — is to subsample this dataset the same way the learning-curve experiments do, and track how the significant-rule count per endpoint degrades with N.